### 1️⃣ Import Libraries

In [111]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

### 2️⃣ Simulate Users and Posts

In [112]:
np.random.seed(42)
num_users = 100
num_posts = 200

# User features
user_df = pd.DataFrame({
    'user_id': np.arange(num_users),
    'age': np.random.randint(13, 65, size=num_users),
    'gender': np.random.randint(0, 2, size=num_users),
    'num_friends': np.random.randint(50, 500, size=num_users),
    'avg_likes_received': np.random.rand(num_users) * 100,
    'avg_comments_received': np.random.rand(num_users) * 50,
    'avg_shares_received': np.random.rand(num_users) * 30,
    'active_days_last_week': np.random.randint(0, 7, size=num_users),
    'time_spent_last_week': np.random.rand(num_users) * 1000,
    'num_groups': np.random.randint(1, 20, size=num_users),
    'has_profile_picture': np.random.randint(0, 2, size=num_users)
})

# Post features
post_df = pd.DataFrame({
    'post_id': np.arange(num_posts),
    'post_length': np.random.randint(20, 2000, size=num_posts),
    'num_images': np.random.randint(0, 5, size=num_posts),
    'num_videos': np.random.randint(0, 3, size=num_posts),
    'num_hashtags': np.random.randint(0, 10, size=num_posts),
    'author_followers': np.random.randint(100, 10000, size=num_posts),
    'author_following': np.random.randint(50, 5000, size=num_posts),
    'author_posts_last_week': np.random.randint(0, 20, size=num_posts),
    'post_type': np.random.choice(['text', 'image', 'video'], size=num_posts),
    'post_time_hour': np.random.randint(0,24,size=num_posts),
    'is_boosted': np.random.randint(0,2,size=num_posts)
})

### 3️⃣ Feature Engineering

- 3a: Normalize continuous features
- 3b: One-hot encode categorical features

In [113]:
user_cont_features = ['age','num_friends','avg_likes_received','avg_comments_received',
                      'avg_shares_received','active_days_last_week','time_spent_last_week','num_groups']
scaler = StandardScaler()
user_df[user_cont_features] = scaler.fit_transform(user_df[user_cont_features])
user_df['has_profile_picture'] = user_df['has_profile_picture'].astype(float)

post_cont_features = ['post_length','num_images','num_videos','num_hashtags',
                      'author_followers','author_following','author_posts_last_week']
post_df[post_cont_features] = scaler.fit_transform(post_df[post_cont_features])

In [114]:
post_df = pd.get_dummies(post_df, columns=['post_type','post_time_hour'])

### 4️⃣ Correlation Analysis
- HeatMap
- Variance Inflation Factor (VIF)</br>
📈 Range and Interpretation</br>

| **VIF value** | **Meaning** | **Interpretation** |
|:----------------|:-------------:|--------------------:|
| = 1 | No correlation | The feature is independent |
| 1–5 | Moderate correlation | Usually acceptable |
| > 5 | High correlation | Investigate for redundancy |
| > 10 | Very high correlation | Likely problematic |


In [ ]:
# User-only correlation heatmap & VIF
user_corr = user_df[user_cont_features + ['has_profile_picture']].corr()
plt.figure(figsize=(10,8))
sns.heatmap(user_corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("User feature correlation")
plt.show()

# VIF for user features
X_user = user_df[user_cont_features + ['has_profile_picture']].values
vif_user = pd.DataFrame({
    'feature': user_cont_features + ['has_profile_picture'],
    'VIF': [variance_inflation_factor(X_user, i) for i in range(X_user.shape[1])]
})
print(vif_user)


# Post-only correlation heatmap & VIF
post_corr = post_df[post_cont_features].corr()
plt.figure(figsize=(10,8))
sns.heatmap(post_corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Post feature correlation")
plt.show()

# VIF for post features
X_post = post_df[post_cont_features].values
vif_post = pd.DataFrame({
    'feature': post_cont_features,
    'VIF': [variance_inflation_factor(X_post, i) for i in range(X_post.shape[1])]
})
print(vif_post)

## 5️⃣ Create Interaction DataFrame

In [116]:
interactions = []
for user_id in range(num_users):
    liked_posts = np.random.choice(num_posts, size=20, replace=False)
    for post_id in range(num_posts):
        interactions.append({
            'user_id': user_id,
            'post_id': post_id,
            'label': 1 if post_id in liked_posts else 0
        })
df = pd.DataFrame(interactions)
df = df.merge(user_df, on='user_id', how='left')
df = df.merge(post_df, on='post_id', how='left')
df = df.reset_index(drop=True)

In [117]:
df.head(5)

,user_id,post_id,label,age,gender,num_friends,avg_likes_received,avg_comments_received,avg_shares_received,active_days_last_week,...,post_time_hour_14,post_time_hour_15,post_time_hour_16,post_time_hour_17,post_time_hour_18,post_time_hour_19,post_time_hour_20,post_time_hour_21,post_time_hour_22,post_time_hour_23
0,0,0,0,0.853003,0,0.836003,-1.266711,0.160569,-0.648395,-0.394709,...,False,False,True,False,False,False,False,False,False,False
1,0,1,0,0.853003,0,0.836003,-1.266711,0.160569,-0.648395,-0.394709,...,False,False,False,False,False,False,False,False,False,False
2,0,2,1,0.853003,0,0.836003,-1.266711,0.160569,-0.648395,-0.394709,...,False,False,False,False,False,False,False,False,False,False
3,0,3,0,0.853003,0,0.836003,-1.266711,0.160569,-0.648395,-0.394709,...,False,False,False,False,False,True,False,False,False,False
4,0,4,1,0.853003,0,0.836003,-1.266711,0.160569,-0.648395,-0.394709,...,False,True,False,False,False,False,False,False,False,False


## 6️⃣ Convert to PyTorch tensors

In [130]:
user_feat_cols = user_cont_features + ['has_profile_picture']
post_feat_cols = [c for c in df.columns if c not in (user_feat_cols + ['user_id','post_id','label'])]
df[post_feat_cols] = df[post_feat_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

user_features = torch.tensor(df[user_feat_cols].to_numpy(dtype=np.float32))
post_features = torch.tensor(df[post_feat_cols].to_numpy(dtype=np.float32))
labels = torch.tensor(df['label'].values, dtype=torch.float32)

## 7️⃣ Define the Two-Tower Model


In [119]:
class TwoTowerRecommender(nn.Module):
    def __init__(self, user_feat_dim, post_feat_dim, hidden_dim=32, dropout=0.2):
        super().__init__()
        # User tower
        self.user_tower = nn.Sequential(
            nn.Linear(user_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # Post tower
        self.post_tower = nn.Sequential(
            nn.Linear(post_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
    def forward(self, user_feat, post_feat):
        user_emb = self.user_tower(user_feat)      # shape: [batch, hidden_dim]
        post_emb = self.post_tower(post_feat)      # shape: [batch, hidden_dim]
        scores = (user_emb * post_emb).sum(dim=1)  # dot product
        return torch.sigmoid(scores)               # probability of engagement

## 8️⃣ Train/Validation Split

In [120]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(range(len(labels)), test_size=0.2, random_state=42)

train_user_idx = train_idx
test_user_idx = test_idx

train_user_feat = user_features[train_user_idx]
train_post_feat = post_features[train_user_idx]
train_labels = labels[train_user_idx]

test_user_feat = user_features[test_user_idx]
test_post_feat = post_features[test_user_idx]
test_labels = labels[test_user_idx]

## 9️⃣ Training Loop

In [121]:
# Hyperparameters
hidden_dim = 64
dropout = 0.2
lr = 0.02
epochs = 20

model = TwoTowerRecommender(user_feat_dim=train_user_feat.shape[1],
                            post_feat_dim=train_post_feat.shape[1],
                            hidden_dim=hidden_dim, dropout=dropout)

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.BCELoss()

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    preds = model(train_user_feat, train_post_feat)
    loss = criterion(preds, train_labels)
    
    loss.backward()
    optimizer.step()
    
    # Validation
    model.eval()
    with torch.no_grad():
        preds = model(test_user_feat, test_post_feat)
        test_loss = criterion(preds, test_labels)
    
    print(f"Epoch {epoch+1}: train_loss={loss.item():.4f}, val_loss={test_loss.item():.4f}")

Epoch 1: train_loss=0.9204, val_loss=0.6938
Epoch 2: train_loss=0.6951, val_loss=0.6931
Epoch 3: train_loss=0.6932, val_loss=0.6931
Epoch 4: train_loss=0.6931, val_loss=0.6931
Epoch 5: train_loss=0.6931, val_loss=0.6931
Epoch 6: train_loss=0.6931, val_loss=0.6931
Epoch 7: train_loss=0.6931, val_loss=0.6931
Epoch 8: train_loss=0.6931, val_loss=0.6931
Epoch 9: train_loss=0.6931, val_loss=0.6931
Epoch 10: train_loss=0.6931, val_loss=0.6931
Epoch 11: train_loss=0.6931, val_loss=0.6931
Epoch 12: train_loss=0.6931, val_loss=0.6931
Epoch 13: train_loss=0.6931, val_loss=0.6931
Epoch 14: train_loss=0.6931, val_loss=0.6931
Epoch 15: train_loss=0.6931, val_loss=0.6931
Epoch 16: train_loss=0.6931, val_loss=0.6931
Epoch 17: train_loss=0.6931, val_loss=0.6931
Epoch 18: train_loss=0.6931, val_loss=0.6931
Epoch 19: train_loss=0.6931, val_loss=0.6931
Epoch 20: train_loss=0.6931, val_loss=0.6931
